# Paper tables

Variant A uses exterior Q only. Variant B adds interior Q, floor-plan Q, and mean unfinished probability. Both use the photo offer recorded for each property in the archived source links.

In [1]:
from pathlib import Path
import sys

working_dir = Path.cwd()
root = next(
    (path for path in (working_dir, *working_dir.parents) if (path / "inputs" / "prompts.json").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Open this notebook from the final_project directory or one of its subdirectories.")
sys.path.insert(0, str(root / "code"))

In [2]:
from IPython.display import display
import pandas as pd

from utils.paper_data import DATASETS_DIR, MARKETS, load_analysis_data, load_llama_scores

tables_dir = root / "outputs" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
listings, photo_types, quality_models_by_type = load_analysis_data(root)


def save_table(table, number, **options):
    display(table)
    path = tables_dir / f"table_{number}.tex"
    path.write_text(table.to_latex(float_format="{:.3f}".format, escape=True, na_rep="", **options), encoding="utf-8")

### Table 2. Sample means

In [3]:
categorical = {
    "roomsCount_grouped": "Number of rooms",
    "bathrooms_cat": "Number of bathrooms",
    "ceilingHeightCapped": "Ceiling height (m)",
    "apartment": "Apartment",
    "toDemolish": "Demolition program",
    "floorNumberCapped": "Floor number",
    "material": "Construction material",
    "repairType": "State of repair",
}
numerical = {
    "totalArea": "Area (square metres)",
    "age": "Building age (years)",
    "distance_center": "Distance to center",
    "distance_metro": "Distance to metro",
    "distance_nice": "Distance to green area or water",
    "distance_ugly": "Distance to industrial zone",
}
means = {}
for market, title in MARKETS.items():
    data = listings.loc[listings["deal_type"].eq(market) & listings["link_num"].gt(0)]
    rows = {(label, "Mean"): data[column].mean() for column, label in numerical.items()}
    for column, label in categorical.items():
        shares = data[column].value_counts(normalize=True, sort=False)
        if column in ("apartment", "toDemolish"):
            shares = shares.loc[["yes"]]
        rows.update({(label, level): share for level, share in shares.items()})
    rows["Listings with usable photos", "N"] = f"{len(data):,}"
    means[title] = pd.Series(rows)

table = pd.DataFrame(means)
table.index.names = ["Variable", "Category"]
# Geographic distances are in units of 100 metres.
save_table(table, "02")

Long-term Rentals Secondary Sales
Variable                        Category                                  
Area (square metres)            Mean             54.852406       61.845447
Building age (years)            Mean             32.637999       29.612184
Distance to center              Mean            111.031193      109.099358
Distance to metro               Mean              7.032753        7.212756
Distance to green area or water Mean              7.081138        6.730674
Distance to industrial zone     Mean              5.598996        5.414793
Number of rooms                 1                 0.381044        0.286459
                                2                 0.403759        0.380358
                                3                 0.181032         0.26076
                                4+                0.034165        0.072423
Number of bathrooms             1                  0.87062        0.787762
                                2+                 0.12938        0.212238
Ceiling height (m)              2.4-2.7           0.178667        0.239464
                                2.7-3.0            0.15477         0.19283
                                3.0+              0.106603         0.26112
                                NR                 0.55996        0.306586
Apartment                       yes                0.03877        0.064876
Demolition program              yes               0.051714        0.022823
Floor number                    1st                0.05047        0.055081
                                2-9               0.606012        0.578399
                                10-19             0.265044        0.264983
                                20-29             0.057315        0.067212
                                30+               0.021159        0.034325
Construction material           brick             0.164665         0.16165
                                block             0.080839        0.071974
                                panel             0.330948        0.305598
                                concrete          0.225714        0.353581
                                NR                0.197834        0.107197
State of repair                 Designer          0.225278        0.228682
                                Euro              0.381106        0.210621
                                Cosmetic          0.301263        0.300297
                                None              0.011264        0.188337
                                NR                0.081088        0.072064
Listings with usable photos     N                   16,069          11,129

### Table A.1. LLaMA4 image-rating prompt

In [4]:
prompts = pd.Series({
    "System prompt": "You are a helpful assistant that can look at pictures of real estate and provide accurate descriptions and analysis. Answer the user's questions regarding the attached images as clearly and accurately as possible.",
    "User query": "Rate how luxurious and upscale this apartment looks based on the pictures. Give a score from 0 to 10 to each photo. Answer with a list of scores (e.g.: '[4,3,5,7,2,0,9]'). Omit any explanation or additional information.",
    "Image inputs": "Up to ten photo URLs, randomly selected when a listing has more than ten photos.",
    "Decoding": "Temperature 0",
}, name="Prompt or setting")
save_table(prompts.to_frame(), "A1", column_format=r"lp{0.73\linewidth}")

,Prompt or setting
System prompt,You are a helpful assistant that can look at p...
User query,Rate how luxurious and upscale this apartment ...
Image inputs,"Up to ten photo URLs, randomly selected when a..."
Decoding,Temperature 0


## Score variants

The variant tables use each model's eligible sample. The two shared-sample comparisons use identical listings within each market and outcome.

In [5]:
from utils.paper_variant_tables import export_variant_tables

llama_path = root / DATASETS_DIR / "llama4_scores.parquet"
llama = load_llama_scores(llama_path, listings) if llama_path.is_file() else None
shared_price, shared_duration = export_variant_tables(
    tables_dir, listings, photo_types, quality_models_by_type, llama,
)
display(shared_price.round(3))
display(shared_duration.round(3))

Saved Tables 01 and 03–07 for variant A
Saved Tables 01 and 03–07 for variant B


,Market,Variant,Train N,Test N,OLS adjusted R²,OLS RMSE,Boosting adjusted R²,Boosting RMSE
0,Long-term Rentals,A,3317,830,0.887,0.163,0.903,0.165
2,Long-term Rentals,B,3317,830,0.902,0.151,0.912,0.157
1,Secondary Sales,A,4628,1157,0.905,0.180,0.925,0.170
3,Secondary Sales,B,4628,1157,0.910,0.175,0.927,0.167


,Market,Variant,Score,N,Closed,Censored,Concordance,HR (+1 point),Lower 95%,Upper 95%
0,Long-term Rentals,A,Adjusted exterior Q-score,4139,3995,144,0.619,0.973,0.861,1.101
2,Long-term Rentals,B,Adjusted interior Q-score,4139,3995,144,0.623,1.590,1.253,2.019
3,Long-term Rentals,B,Adjusted exterior Q-score,4139,3995,144,0.623,0.927,0.818,1.051
4,Long-term Rentals,B,Adjusted floor-plan Q-score,4139,3995,144,0.623,0.953,0.764,1.190
5,Long-term Rentals,B,Mean unfinished probability,4139,3995,144,0.623,0.919,0.631,1.338
1,Secondary Sales,A,Adjusted exterior Q-score,5766,4755,1011,0.617,1.164,1.020,1.327
6,Secondary Sales,B,Adjusted interior Q-score,5766,4755,1011,0.618,1.231,1.076,1.408
7,Secondary Sales,B,Adjusted exterior Q-score,5766,4755,1011,0.618,1.110,0.970,1.269
8,Secondary Sales,B,Adjusted floor-plan Q-score,5766,4755,1011,0.618,1.057,0.862,1.295
9,Secondary Sales,B,Mean unfinished probability,5766,4755,1011,0.618,0.829,0.694,0.990


## Listing and photo coverage

In [6]:
from utils.paper_variants import VARIANTS

rows = []
for market in MARKETS:
    data = listings.loc[listings["deal_type"].eq(market)]
    metrics = {
        "listings": len(data),
        "first_listing_date": data["date0"].min().date().isoformat(),
        "last_listing_date": data["date0"].max().date().isoformat(),
        "last_observed_date": data["date1"].max().date().isoformat(),
        "mean_new_listings_per_day": data.groupby(data["date0"].dt.date).size().mean(),
        "owner_share": data["author_cat"].eq("Owner").mean(),
        "listings_with_usable_photos": data["link_num"].gt(0).sum(),
        "fraction_without_usable_photos": data["link_num"].eq(0).mean(),
        "usable_photos": data["link_num"].sum(),
        "listings_with_adjusted_interior_score": data["clip_polarity_adj"].count(),
        "listings_with_exterior_score": data["exterior_polarity_adj"].count(),
        "listings_with_floor_plan_score": data["plan_polarity_adj"].count(),
        "listings_with_unfinished_probability": data["mean_p_unfinished"].count(),
        "listings_with_all_clip_features": len(data.dropna(subset=list(VARIANTS["B"]))),
    }
    for kind, photos in photo_types.items():
        metrics[f"{kind}_photos"] = photos["deal_type"].eq(market).sum()
    rows.extend((metric, market, value) for metric, value in metrics.items())

numbers = pd.DataFrame(rows, columns=["metric", "deal_type", "value"])
numbers.to_csv(tables_dir / "paper_numbers.csv", index=False)
display(numbers.pivot(index="metric", columns="deal_type", values="value"))

deal_type,long_rent,sale_secondary
metric,,
exterior_photos,40450,58116
first_listing_date,2025-07-01,2025-07-01
fraction_without_usable_photos,0.06841,0.054621
interior_photos,229659,150243
last_listing_date,2025-08-14,2025-08-14
last_observed_date,2025-12-05,2025-12-05
listings,17249,11772
listings_with_adjusted_interior_score,15947,10477
listings_with_all_clip_features,4189,5844
